# Reproducing Skromne Carrasco, Viejo & Peyrache (2026) from dandiset 001676

Paper: "Months-long stability of the head-direction system", Nature 652, 167-173 (2026),
doi:10.1038/s41586-025-10096-w. Claims: `claims/001676/10.1038-s41586-025-10096-w.yaml`, written
from the version of record (main text, Methods, Extended Data legends; the Supplementary
Information was not read).

Dataset: DANDI:001676, version 0.251205.2137 (doi:10.48324/dandi.001676/0.251205.2137). This is
the version the paper's Data availability statement cites, and the only published version; the
draft has the same 406 assets and size. Files are streamed; nothing is written to disk.

The dataset holds 402 Miniscope sessions of ten mice in four protocols, and one
electrophysiology session of each of four mice. This notebook tests the imaging and
electrophysiology claims except the decoders, which are in `decoding.ipynb`.

| Claim | Where | Status |
|---|---|---|
| roi-counts | here | tested |
| edf1e-hd-proportion | here | tested |
| edf1h-ring | here | tested |
| fig1f-example-stable, fig1g-all-mice-stable, edf4ab-vtest-per-animal, edf4c-previous-session | here | tested |
| edf2ab-jaccard, edf2-no-registration-confound | here | tested |
| fig1h-drift-rate, edf4ef-not-diffusive, edf4f-change-below-shuffle | here | tested |
| edf4g-pairwise-offset-stable, fig1i-offset-drift, edf4ij-pv-decay | here | tested |
| fig1k-decoding-across-days | decoding.ipynb | blocked, too_large (1,883 decoder fits, estimated at 80 min of fits in decoding.ipynb) |
| fig2e-offset-across-environments, edf5c-stable-within-environment | here | tested |
| fig2g-environment-classifier, fig2i-random-remapping, fig2h-coherent-remapping | here | tested |
| edf7-ephys-offsets | here | tested |
| edf7g-waveforms | | blocked, data_missing (no waveforms in the files) |
| edf8-same-vs-mixed | decoding.ipynb | tested |
| fig3de-probe-memory, fig3e-mvl-delay, fig3fg-probe-drift | | blocked, metadata_insufficient (no registration between the two-week protocol and the probe week) |
| edf9-interference | here | tested |
| fig4d-single-visit-memory, edf10c-environment-identity, edf10b-per-animal-change | here | tested |
| fig4e-decoder-single-visit | decoding.ipynb | tested |

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import hdstab as H

T0 = time.time()
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DEG = np.degrees
print("dandiset", H.DANDISET_ID, "version", H.DANDISET_VERSION)

## Loading

Every imaging session is read once and reduced to its tuning curves (raw, and smoothed under
both readings of the kernel), split-half correlations and HDI per ROI, its CellReg table, and,
for the first and last sessions of the circular protocol, the ROI centroids. This is most of
the run time, nearly all of it streaming; the time is printed below.

In [ ]:
ASSETS = H.list_assets()
IMAGING = [a for a in ASSETS if "ophys" in a["path"]]
EPHYS = [a for a in ASSETS if "ecephys" in a["path"]]
print(len(ASSETS), "assets:", len(IMAGING), "imaging,", len(EPHYS), "electrophysiology")
S = H.load_sessions(IMAGING, masks=True, workers=12)
print("loaded", len(S), "imaging sessions in %.0f s" % (time.time() - T0))

In [ ]:
EP = [H.load_ephys(a) for a in EPHYS]
print("loaded", len(EP), "electrophysiology sessions")

## Data check

Counts by mouse and protocol, the ranges and units of the variables used, the CellReg tables,
the stored tuning curves, and the clock shared by dF/F and head direction.

In [ ]:
info = pd.DataFrame([{k: s[k] for k in ["mouse", "protocol", "env", "k", "n", "start", "nroi", "nframes", "fs", "duration_s"]}
                     for s in S])
print("sessions by mouse and protocol")
print(pd.crosstab(info.mouse, info.protocol, margins=True))
print("\nsessions by protocol and environment")
print(pd.crosstab(info.protocol, info.env))
print("\nsession numbers k run 1..n without gaps:",
      all(sorted(g.k) == list(range(1, g.n.iloc[0] + 1)) for _, g in info.groupby(["mouse", "protocol"])))
dates_ok = []
for (m, p), g in info.groupby(["mouse", "protocol"]):
    d = pd.to_datetime(g.sort_values("k").start.str[:19]).values
    dates_ok.append(bool((np.diff(d) > np.timedelta64(0)).all()))
print("start times increase with k in every protocol:", all(dates_ok))
print("\nimaging rate (Hz) %.2f to %.2f; frames %d to %d; duration %.0f to %.0f s"
      % (info.fs.min(), info.fs.max(), info.nframes.min(), info.nframes.max(), info.duration_s.min(), info.duration_s.max()))
hdr = np.array([s["hd_range"] for s in S]); dfr = np.array([s["dff_range"] for s in S])
print("head direction (radians) min %.4f, max %.4f; dF/F (arbitrary) min %.3f, max %.3f"
      % (hdr[:, 0].min(), hdr[:, 1].max(), dfr[:, 0].min(), dfr[:, 1].max()))
print("ROIs per session %d to %d" % (info.nroi.min(), info.nroi.max()))
print("\nCellReg tables")
reg_rows = []
for (m, p), _ in info.groupby(["mouse", "protocol"]):
    r = H.registration_check(H.group(S, m, p))
    reg_rows.append(dict(mouse=m, protocol=p, **r))
print(pd.DataFrame(reg_rows).to_string(index=False))
print("\nCellReg scores (a CellReg score is a confidence, 0 to 1): rows with a score below 0")
neg_rows = []
for (m, p), _ in info.groupby(["mouse", "protocol"]):
    sc = H.group(S, m, p)[0]["cellreg_score"]
    neg_rows.append(dict(mouse=m, protocol=p, rows=len(sc), below_0=int((sc < 0).sum()), min=round(float(sc.min()), 3), max=round(float(sc.max()), 3)))
neg_rows = pd.DataFrame(neg_rows)
print(neg_rows.to_string(index=False))
print("total rows with a score below 0:", int(neg_rows.below_0.sum()), "of", int(neg_rows.rows.sum()))

In [ ]:
# The stored tuning curves (processing/ophys/tuning_curves) against the ones computed here.
rows = []
for s in S:
    st = s["tc_stored"].astype(float)
    cc = np.corrcoef(st) if len(st) > 1 else np.ones((1, 1))
    same = float(np.nanmin(cc[np.triu_indices(len(st), 1)])) if len(st) > 1 else 1.0
    best = []
    for name, sig in [("raw", 0), ("s2", 2), ("s4 (12 deg)", 4), ("s6", 6), ("s8", 8), ("s12 (12 bins)", 12)]:
        x = H.fill_nan_circ(s["tc_raw"].astype(float)) if sig == 0 else H.smooth(s["tc_raw"].astype(float), sig)
        c = H.row_corr(x, np.tile(st[0], (len(x), 1)))
        best.append((name, int(np.nanargmax(c)), float(np.nanmax(c))))
    top = max(best, key=lambda b: b[2])
    rows.append(dict(mouse=s["mouse"], protocol=s["protocol"], min_row_corr=same, best_roi=top[1],
                     best_kernel=top[0], best_r=top[2], r_roi0_s4=[b[2] for b in best if b[0].startswith("s4")][0]))
stored = pd.DataFrame(rows)
print("smallest correlation between any two stored rows of a session: median %.4f, min %.4f"
      % (stored.min_row_corr.median(), stored.min_row_corr.min()))
print("sessions whose stored curve best matches computed ROI 0: %d of %d" % ((stored.best_roi == 0).sum(), len(stored)))
print("kernel giving the best match (sessions):", stored.best_kernel.value_counts().to_dict())
print("correlation with ROI 0 smoothed at 4 bins: median %.3f" % stored.r_roi0_s4.median())

Clock check. dF/F and head direction carry their own timestamps. If they share a clock, head
direction tuning is strongest with no shift between them. One session per mouse of the circular
protocol (the session with the most ROIs passing the split-half rule, also used for the Isomap
claim) is read in full, and the median HDI and split-half correlation over its ROIs are computed
with head direction shifted by -5 to +5 s. The electrophysiology files are checked the same way
below (edf7 section).

In [ ]:
EXAMPLE = {}
for m in range(1, 11):
    g = [s for s in S if s["mouse"] == m]
    best = max(g, key=lambda s: (s["split_12deg"] > 0.8).sum())
    EXAMPLE[m] = best
RAW = {m: H.load_raw(next(a for a in IMAGING if a["path"] == EXAMPLE[m]["path"])) for m in range(1, 6)}
shifts = [-5, -2, -1, -0.5, -0.2, 0, 0.2, 0.5, 1, 2, 5]
clock = []
for m in range(1, 6):
    dff, ts, hd, hts = RAW[m]
    print("mouse %d, %s session %d: dF/F %.2f to %.2f s, head direction %.2f to %.2f s"
          % (m, EXAMPLE[m]["protocol"], EXAMPLE[m]["k"], ts[0], ts[-1], hts[0], hts[-1]))
    for sh, h, c in H.shifted_hdi(dff, ts, hd, hts, shifts):
        clock.append(dict(mouse=m, shift_s=sh, median_hdi=h, median_split=c))
clock = pd.DataFrame(clock)
print(clock.pivot(index="shift_s", columns="mouse", values="median_hdi").round(4))
print("shift with the highest median HDI per mouse:", clock.loc[clock.groupby("mouse").median_hdi.idxmax()].shift_s.tolist())

## HD cells

The HD cell rule of the Methods, under the readings the claims file leaves open: the kernel
(12 degrees or 12 bins) and whether HDI is computed on the raw or the smoothed tuning curve.
The main reading is 12 degrees with HDI on the smoothed curve, an assumption. The stored tuning
curves (data check above) match the computed ones best with a kernel of 4 bins (12 degrees).
For the HDI, this reading gives the numbers of sessions the paper compares (18 for the Fig. 1f
example, 84 pooled in Fig. 1g; 16, 18, 16, 16 in Extended Data Fig. 4a) and of offset values
(25,901 in Extended Data Fig. 4g) most closely; the counts are printed under each claim. The
numbers of sessions implied by the Wilcoxon p values of Extended Data Fig. 4f (16, 18, 18, 16, 16)
also fit it (16, 18, 18, 16, 17 here). It conflicts with other counts: it keeps few cells per
session (printed below), its HD proportions are 3 to 10 times below those of Extended Data
Fig. 1e, and raw HDI comes closer to the paper's 125 HD cells of Extended Data Fig. 7. The
paper's 72 cells of Extended Data Fig. 2 lie between the two readings. Every claim below that rests on HD cells, except the ring of
edf1h-ring, is also run with raw HDI (12 degree kernel), and the claims of Fig. 1 under all four
readings.

In [ ]:
CONFIGS = [("12deg", "smooth"), ("12deg", "raw"), ("12bins", "raw"), ("12bins", "smooth")]
MAIN = CONFIGS[0]
MICE15 = [1, 2, 3, 4, 5]


def tables(protocol, cfg=MAIN, **kw):
    out = {}
    for m in range(1, 11):
        g = H.group(S, m, protocol)
        if g:
            out[m] = H.hd_table(g, sigma=cfg[0], hdi_kind=cfg[1], **kw)
    return out


CIRC = {cfg: tables("circular", cfg) for cfg in CONFIGS}
rows = []
for cfg in CONFIGS:
    for m, T in CIRC[cfg].items():
        rows.append(dict(reading="/".join(cfg), mouse=m, hd_cells=int(T["valid"].any(1).sum()),
                         median_per_session=float(np.median(T["valid"].sum(0)))))
print("HD cells of the circular protocol (CellReg score > 0.8, registered in >= 6 sessions)")
print(pd.DataFrame(rows).pivot(index="reading", columns="mouse", values="hd_cells"))
print("\nmedian HD cells per session")
print(pd.DataFrame(rows).pivot(index="reading", columns="mouse", values="median_per_session"))

## roi-counts

> Across 388 imaging sessions, the number of ROIs per session ranged from 21 to 286 (95 plus or
> minus 56, mean and s.d.), and was stable over time.

In [ ]:
for label, sel in [("(a) all imaging sessions", info), ("(b) mice 1 to 5", info[info.mouse <= 5])]:
    print("%s: %d sessions, ROIs %d to %d, mean %.1f, s.d. %.1f"
          % (label, len(sel), sel.nroi.min(), sel.nroi.max(), sel.nroi.mean(), sel.nroi.std(ddof=0)))
print("expected 388 sessions, 21 to 286, 95 +- 56")
print("\nSpearman correlation of ROI count with session number, circular protocol")
sp = []
for m in MICE15:
    g = info[(info.mouse == m) & (info.protocol == "circular")].sort_values("k")
    r, p = stats.spearmanr(g.k, g.nroi)
    sp.append(dict(mouse=m, n=len(g), rho=r, p=p))
sp = pd.DataFrame(sp)
print(sp.round(4).to_string(index=False))
print("mice with p < 0.05: %d of 5" % (sp.p < 0.05).sum())
print("\nother protocols (variant):")
for p in ["multi", "random", "single"]:
    out = []
    for m in sorted(info[info.protocol == p].mouse.unique()):
        g = info[(info.mouse == m) & (info.protocol == p)].sort_values("k")
        r, pv = stats.spearmanr(g.k, g.nroi)
        out.append("m%d rho %.2f p %.3f" % (m, r, pv))
    print(p, "; ".join(out))

**Result: partially reproduced.** The dataset holds 402 imaging sessions, 367 of them from mice 1 to 5, and neither count is the paper's 388. ROIs per session range from 22 to 291 (mean 102.1, s.d. 60.6) over all sessions and from 26 to 286 (98.9, 56.7) for mice 1 to 5; both are within tolerance of 21 to 286 and 95. ROI count is correlated with session number at p < 0.05 in 2 of 5 mice (mouse 3, rho -0.87; mouse 4, rho 0.63), so a majority show no trend.

## edf1e-hd-proportion

> In the four-week circular protocol, the median proportion of ROIs detected as HD cells per
> animal was 0.2, 0.7, 0.4, 0.7 and 0.2, with 72, 42, 197, 48 and 102 ROIs per session on
> average, over 19, 20, 19, 19 and 20 recordings.

Reading (a) counts the HD cells kept for analysis (CellReg score > 0.8, registered in >= 6
sessions); reading (b) applies only the split-half, session HDI and average HDI rules to every
ROI of the session, the average HDI taken over the CellReg row where the ROI is registered and
its own session otherwise.

In [ ]:
EXP_P = [0.2, 0.7, 0.4, 0.7, 0.2]; TOL_P = [0.03, 0.04, 0.05, 0.05, 0.04]
EXP_N = [72, 42, 197, 48, 102]; TOL_N = [16, 5, 18, 12, 25]; EXP_REC = [19, 20, 19, 19, 20]


def proportion_b(g, sigma, hdi_kind):
    T = H.hd_table(g, sigma=sigma, hdi_kind=hdi_kind, score_min=-np.inf, min_sessions=0)
    props = []
    for j, s in enumerate(g):
        hkey = "hdi_raw" if hdi_kind == "raw" else f"hdi_s_{sigma}"
        split, hv = s[f"split_{sigma}"], s[hkey]
        avg = hv.copy()
        col = T["R"][:, j]
        with np.errstate(invalid="ignore"):
            row_avg = np.nanmean(np.where(T["passes"], T["hdi"], np.nan), axis=1)
        for i in np.where(col >= 0)[0]:
            if np.isfinite(row_avg[i]):
                avg[col[i]] = row_avg[i]
        hd = (split >= H.SPLIT_MIN) & (hv >= H.HDI_SESSION_MIN) & (avg > H.HDI_AVG_MIN)
        props.append(hd.mean())
    return np.array(props)


rows = []
for cfg in CONFIGS:
    for m in MICE15:
        g = H.group(S, m, "circular")
        nroi = np.array([s["nroi"] for s in g])
        pa = np.median(CIRC[cfg][m]["valid"].sum(0) / nroi)
        pb = np.median(proportion_b(g, *cfg))
        rows.append(dict(reading="/".join(cfg), mouse=m, a=pa, b=pb, expected=EXP_P[m - 1], tol=TOL_P[m - 1]))
prop = pd.DataFrame(rows)
prop["a_ok"] = (prop.a - prop.expected).abs() <= prop.tol
prop["b_ok"] = (prop.b - prop.expected).abs() <= prop.tol
print(prop.round(3).to_string(index=False))
print("\nproportions within tolerance, of 5:")
print(prop.groupby("reading")[["a_ok", "b_ok"]].sum())
rr = []
for m in MICE15:
    g = info[(info.mouse == m) & (info.protocol == "circular")]
    rr.append(dict(mouse=m, recordings=len(g), expected_rec=EXP_REC[m - 1], roi_mean=g.nroi.mean(), roi_sd=g.nroi.std(ddof=0),
                   expected_mean=EXP_N[m - 1], tol=TOL_N[m - 1], within=abs(g.nroi.mean() - EXP_N[m - 1]) <= TOL_N[m - 1]))
print(pd.DataFrame(rr).round(1).to_string(index=False))
print("variant, average HDI over all registered sessions (main kernel, raw HDI), reading (a):",
      [round(float(np.median(H.hd_table(H.group(S, m, "circular"), avg="all")["valid"].sum(0)
                             / np.array([s["nroi"] for s in H.group(S, m, "circular")]))), 3) for m in MICE15])

**Result: not reproduced.** The recording counts match (19, 20, 19, 19, 20). The mean ROIs per session are within the stated s.d. for mice 1, 2 and 4 (83.4, 45.3, 58.4) and above it for mice 3 and 5 (238.8 against 197 +- 18, 129.8 against 102 +- 25). Under any reading, at most 1 of the 5 median HD proportions is within tolerance; under the main reading (a) they are 0.116, 0.400, 0.035, 0.292 and 0.072, against 0.2, 0.7, 0.4, 0.7 and 0.2.

## Change in preferred direction, circular protocol

For each session, the circular mean over HD cells valid in it and in the reference session of
each cell's shift (cross-correlation of tuning curves), against the last session or the previous
one; shuffles permute cell identity in the compared session, 1,000 times (Methods, Orientation
remapping). Computed under the four readings; shuffles under the main one.

In [ ]:
N_SHUF = 1000


def changes_vs(T, ref, rng):
    """Session changes against `ref` ("last" or "previous"), shuffled circular means (sessions x 1000)
    and per-session per-cell shifts."""
    ns = T["valid"].shape[1]
    out = []
    for j in range(ns):
        r = ns - 1 if ref == "last" else j - 1
        if r < 0 or r == j:
            continue
        cells, M = H.shift_matrix(T, r, T, j)
        if len(cells) == 0:
            continue
        own = np.diag(M)
        sh_mean, _ = H.shuffled_means(M, rng, N_SHUF)
        out.append(dict(j=j, n_cells=len(cells), change=H.circ_mean(own), cell_shifts=own, shuffle=sh_mean))
    return out


CH = {}
for cfg in CONFIGS:
    for ref in ["last", "previous"]:
        rng = np.random.default_rng(1)
        CH[(cfg, ref)] = {m: changes_vs(CIRC[cfg][m], ref, rng) for m in MICE15}
for cfg in CONFIGS:
    print("/".join(cfg), "sessions compared to the last:", [len(CH[(cfg, "last")][m]) for m in MICE15],
          "to the previous:", [len(CH[(cfg, "previous")][m]) for m in MICE15])
print("HD cells per compared session (main reading, against last): median",
      [int(np.median([x["n_cells"] for x in CH[(MAIN, "last")][m]])) for m in MICE15],
      "min", [min(x["n_cells"] for x in CH[(MAIN, "last")][m]) for m in MICE15])


def vstats(cfg, ref):
    rows = []
    for m in MICE15:
        c = np.array([x["change"] for x in CH[(cfg, ref)][m]])
        V, p = H.vtest(c)
        sh = np.array([x["shuffle"] for x in CH[(cfg, ref)][m]])  # sessions x shuffles
        Vsh = np.array([H.vtest(sh[:, k])[0] for k in range(sh.shape[1])])
        rows.append(dict(mouse=m, n=len(c), V=V, p_one_sided=p, p_two_sided=min(1, 2 * p),
                         V_shuffle_median=np.median(Vsh), mad_deg=np.median(np.abs(DEG(c))),
                         circ_mean_deg=DEG(H.circ_mean(c)), sd_deg=np.std(DEG(H.wrap(c)))))
    allc = np.concatenate([[x["change"] for x in CH[(cfg, ref)][m]] for m in MICE15])
    allsh = np.concatenate([np.array([x["shuffle"] for x in CH[(cfg, ref)][m]]) for m in MICE15])
    Vp, pp = H.vtest(allc)
    Vsh = np.median([H.vtest(allsh[:, k])[0] for k in range(allsh.shape[1])])
    return pd.DataFrame(rows), dict(n=len(allc), V=Vp, p_one_sided=pp, V_shuffle_median=Vsh)


VS = {(cfg, ref): vstats(cfg, ref) for cfg in CONFIGS for ref in ["last", "previous"]}

## fig1f-example-stable

> In an example mouse, the population change in preferred direction relative to the last
> session was concentrated around 0 degrees (V-test, v = 17.06, P < 0.001, 18 sessions, median
> absolute deviation 5.0 degrees).

In [ ]:
for cfg in CONFIGS:
    df, pooled = VS[(cfg, "last")]
    print("/".join(cfg))
    print(df[["mouse", "n", "V", "p_one_sided", "p_two_sided", "mad_deg"]].round(4).to_string(index=False))
df = VS[(MAIN, "last")][0]
ok = (df.V - 17.06).abs().le(1.706) & (df.mad_deg - 5.0).abs().le(0.5) & (df.p_two_sided < 0.001)
print("\nmain reading: (a) mouse 4: V %.2f, MAD %.1f deg, p %.2g, within: %s"
      % (df.V[3], df.mad_deg[3], df.p_two_sided[3], bool(ok[3])))
print("(b) mice with 18 compared sessions:", df.mouse[df.n == 18].tolist(), "; any matching:", df.mouse[ok & (df.n == 18)].tolist())
print("any mouse matching V and MAD:", df.mouse[ok].tolist())

**Result: reproduced.** Under the main reading, mouse 2 has 18 compared sessions, V = 17.01, a median absolute deviation of 4.6 degrees and p < 0.001, all within tolerance (reading b). Mouse 4, which Extended Data Fig. 1g names (reading a), gives V = 14.29 and 7.2 degrees, both outside. With raw HDI no mouse with 18 sessions matches; mouse 2 then has 19 sessions, V = 18.35 and 5.3 degrees.

## fig1g-all-mice-stable

> Across all five mice, the population change in preferred direction relative to the last
> session was concentrated around 0 degrees (V-test, v = 75.21, P < 0.001, 84 sessions), with
> per-mouse means of 15.4, 2.1, -5.5, -7.8 and -6.9 degrees.

In [ ]:
EXP_M = [15.4, 2.1, -5.5, -7.8, -6.9]; TOL_M = [24.7, 18.7, 15.2, 23.6, 41.5]
for cfg in CONFIGS:
    df, pooled = VS[(cfg, "last")]
    within = [abs(H.wrap(np.radians(a - e))) <= np.radians(t) for a, e, t in zip(df.circ_mean_deg, EXP_M, TOL_M)]
    print("%-14s pooled V %.2f (n = %d, one-sided p %.2g), shuffle median %.1f; per-mouse circular means %s, s.d. %s; within %d of 5"
          % ("/".join(cfg), pooled["V"], pooled["n"], pooled["p_one_sided"], pooled["V_shuffle_median"],
             np.round(df.circ_mean_deg.values, 1), np.round(df.sd_deg.values, 1), sum(within)))
print("expected pooled V 75.21 +- 7.52, n = 84; means", EXP_M)
df = VS[(MAIN, "last")][0]
rev = [abs(H.wrap(np.radians(-a - e))) <= np.radians(t) for a, e, t in zip(df.circ_mean_deg, EXP_M, TOL_M)]
print("main reading, sign reversed (last session minus compared session): means %s; within %d of 5; signs as the paper's in %d of 5 (as computed: %d of 5)"
      % (np.round(-df.circ_mean_deg.values, 1), sum(rev), int((np.sign(-df.circ_mean_deg.values) == np.sign(EXP_M)).sum()),
         int((np.sign(df.circ_mean_deg.values) == np.sign(EXP_M)).sum())))

**Result: reproduced.** Pooled V = 75.58 over 85 sessions (the paper has 75.21 over 84), one-sided p < 0.001. All five per-mouse circular means are within the paper's s.d. (15.5, 1.1, -7.0, -11.0, -12.7 degrees), each with the paper's sign. Under the other readings the pooled V is 62.4 to 80.0 and all five means hold. With the sign reversed (last session minus compared session), 4 of 5 means are within tolerance and none has the paper's sign.

## edf4ab-vtest-per-animal

> The V statistic of the change in preferred direction relative to the last session was 14.1,
> 17.1, 17.3, 14.6 and 12.2 per animal, against shuffle medians of 3.0, 3.2, 5.7, 4.8 and 2.0, and
> 75.2 pooled against a shuffle median of 18.6.

In [ ]:
EXP_V = [14.1, 17.1, 17.3, 14.6, 12.2]; EXP_VSH = [3.0, 3.2, 5.7, 4.8, 2.0]
for cfg in CONFIGS:
    df, pooled = VS[(cfg, "last")]
    w = (df.V - EXP_V).abs() <= 0.1 * np.array(EXP_V)
    print("%-14s V %s shuffle medians %s; within %d of 5; all above shuffle: %s; pooled shuffle median %.1f (expected 18.6 +- 1.86)"
          % ("/".join(cfg), np.round(df.V.values, 2), np.round(df.V_shuffle_median.values, 2), w.sum(),
             bool((df.V > df.V_shuffle_median).all()), pooled["V_shuffle_median"]))
print("expected V", EXP_V, "shuffle medians", EXP_VSH)
print("one-sided p per mouse (main):", VS[(MAIN, "last")][0].p_one_sided.map("{:.2g}".format).tolist())

**Result: reproduced.** The per-animal V are 13.88, 17.01, 17.06, 14.29 and 13.34, all within 10% of the paper's, and each is above its shuffle median (2.63 to 11.94). The pooled shuffle median is 30.8 against the paper's 18.6, outside its tolerance; the criterion does not use it. With raw HDI, 4 of 5 per-animal V are within tolerance.

## edf4c-previous-session

> The change in preferred direction relative to the previous session was also concentrated
> around 0 degrees across the five animals (V-test, P < 0.001, 75 sessions; pooled V 69.9
> against a shuffle median of 21.9).

In [ ]:
EXP_VP = [12.5, 15.6, 16.7, 12.9, 10.1]
for cfg in CONFIGS:
    df, pooled = VS[(cfg, "previous")]
    w = (df.V - EXP_VP).abs() <= 0.1 * np.array(EXP_VP)
    print("%-14s pooled V %.2f (n %d, one-sided p %.2g), shuffle median %.1f; per mouse %s, within %d of 5"
          % ("/".join(cfg), pooled["V"], pooled["n"], pooled["p_one_sided"], pooled["V_shuffle_median"],
             np.round(df.V.values, 2), w.sum()))
print("expected pooled 69.9 +- 6.99, shuffle 21.9 +- 2.19, per mouse", EXP_VP)

**Result: reproduced.** Pooled V = 70.00 over 78 sessions (the paper has 69.9 over 75), one-sided p < 0.001. Per-animal V are within tolerance in 4 of 5 (12.32, 15.46, 16.72, 12.79, 12.71 against 12.5, 15.6, 16.7, 12.9, 10.1). The pooled shuffle median is 26.6 against 21.9. With raw HDI the pooled V is 80.0, outside tolerance, and only 1 of 5 per-animal V is within.

## edf4f-change-below-shuffle

> The absolute change in overall preferred direction with respect to the last day was smaller
> than for shuffled cell identities in every animal (median 15.9, 5.1, 7.9, 26.3 and 24.0
> degrees; one-tailed Wilcoxon, p from 3.8e-6 to 1.5e-4).

In [ ]:
EXP_F = [15.9, 5.1, 7.9, 26.3, 24.0]
for cfg in CONFIGS:
    rows = []
    for m in MICE15:
        L = CH[(cfg, "last")][m]
        a = np.abs(DEG([x["change"] for x in L]))
        shabs = np.abs(DEG(np.array([x["shuffle"] for x in L])))
        for red, f in [("median", np.median), ("mean", np.mean)]:
            s = f(shabs, axis=1)
            w = stats.wilcoxon(a, s, alternative="less")
            rows.append(dict(mouse=m, shuffle=red, n=len(a), median_abs=np.median(a), W=w.statistic, p=w.pvalue))
    df = pd.DataFrame(rows)
    print("/".join(cfg))
    print(df.pivot(index="mouse", columns="shuffle", values=["median_abs", "W", "p"]).round(5))
    dm = df[df.shuffle == "median"]
    print("medians within 10%%: %d of 5; all p < 0.05 (median, mean shuffle): %s, %s"
          % (((dm.median_abs.values - EXP_F) / np.array(EXP_F)).__abs__().__le__(0.1).sum(),
             bool((df[df.shuffle == "median"].p < 0.05).all()), bool((df[df.shuffle == "mean"].p < 0.05).all())))
print("expected", EXP_F)

**Result: partially reproduced.** Under the main reading the medians are 16.4, 4.6, 9.3, 7.2 and 24.0 degrees, 3 of 5 within 10%. The one-tailed Wilcoxon against the shuffles is significant in 4 of 5 mice; mouse 3 is not (p = 0.079 with the median shuffle, 0.106 with the mean). With raw HDI all five tests are significant, but only 2 of 5 medians are within 10%. No reading meets both parts of the criterion; the effect is present, significant in 4 of 5 mice under the main reading and in 5 of 5 with raw HDI.

## edf4ef-not-diffusive

> The drift in preferred direction was not diffusive (not a random walk) but fluctuated around a
> fixed direction, as shown by the cumulative variance of orientation across days.

The variance of the session-level changes (degrees) over the first k recordings, k = 2..n, with
the changes taken relative to the last day (a) or the first day (b). A random walk predicts a
variance growing in proportion to k.

In [ ]:
first = {}
for cfg in CONFIGS:
    rng = np.random.default_rng(2)
    first[cfg] = {}
    for m in MICE15:
        T = CIRC[cfg][m]
        out = []
        for j in range(1, T["valid"].shape[1]):
            cells, M = H.shift_matrix(T, 0, T, j)
            if len(cells):
                out.append(dict(j=j, change=H.circ_mean(np.diag(M))))
        first[cfg][m] = out
for cfg in CONFIGS:
    res = {}
    for reading in ["a", "b"]:
        n_ok = 0
        detail = []
        for m in MICE15:
            if reading == "a":
                c = DEG(H.wrap(np.array([x["change"] for x in CH[(cfg, "last")][m]])))
            else:
                c = DEG(H.wrap(np.r_[0.0, [x["change"] for x in first[cfg][m]]]))
            if len(c) < 6:
                detail.append("m%d too few sessions (%d)" % (m, len(c)))
                continue
            k = np.arange(2, len(c) + 1)
            v = np.array([np.var(c[:kk]) for kk in k])
            slope = (k[:4] @ v[:4]) / (k[:4] @ k[:4])  # first five points of the curve start at k = 1 (variance 0)
            pred = slope * k[-1]
            n_ok += v[-1] < 0.5 * pred
            detail.append("m%d %.0f/%.0f" % (m, v[-1], pred))
        res[reading] = (n_ok, detail)
    print("%-14s (a) last day: %d of 5 below half the random-walk line [%s]; (b) first day: %d of 5 [%s]"
          % ("/".join(cfg), res["a"][0], ", ".join(res["a"][1]), res["b"][0], ", ".join(res["b"][1])))

**Result: reproduced.** With changes relative to the last day (reading a), the variance at the full number of recordings is below half of what the random-walk line predicts in 4 of 5 mice; mouse 2 is the exception. Relative to the first day (reading b) this holds in 2 of 5, and with raw HDI reading (a) holds in 3 of 5. The two readings differ only in the reference session and in which cells enter each session's change, so the pass rests on noisy session values and is fragile.

## edf2ab-jaccard

> The Jaccard index of HD cell detection between sessions of the circular protocol remained high
> regardless of the delay between sessions.

Presence of HD tuning is the HD cell rule of the main reading (valid in the session). Readings:
(a) ROIs registered in both sessions of the pair, (b) ROIs registered in every session.

In [ ]:
for cfg in [MAIN, CONFIGS[1]]:
    for reading, key in [("a", "pair"), ("b", "all")]:
        n_ok, det = 0, []
        for m in MICE15:
            T = CIRC[cfg][m]
            J = H.jaccard(T, key)
            dd = H.day_diff(T["sessions"])
            iu = np.triu_indices(len(dd), 1)
            x, y = dd[iu], J[iu]
            ux = np.unique(x)
            med = np.array([np.nanmedian(y[x == u]) if np.isfinite(y[x == u]).any() else np.nan for u in ux])
            one = med[ux == 1][0] if (ux == 1).any() else np.nan
            last = med[-1]
            rho, p = stats.spearmanr(ux, med, nan_policy="omit")
            ok = last >= 0.8 * one
            n_ok += bool(ok)
            det.append("m%d: 1 day %.2f, %d days %.2f (ratio %.2f), median over pairs %.2f, rho %.2f p %.3g, rows %d"
                       % (m, one, ux[-1], last, last / one, np.nanmedian(y), rho, p, int(T["reg"].all(1).sum()) if key == "all" else -1))
        print("/".join(cfg), "reading", reading, ": longest >= 80% of 1 day in", n_ok, "of 5")
        for d in det:
            print("   ", d)

**Result: not reproduced.** Under the main reading, the median Jaccard index at the longest interval is at least 80% of the one-day value in 2 of 5 mice (reading a) and in 3 of 5 (reading b; mice 4 and 5 have too few ROIs registered in every session to compute it). With raw HDI, reading (a) holds in 5 of 5 mice, so the outcome rests on the HDI reading.

## edf2-no-registration-confound

> The shift in preferred direction between the first and last recording was not significantly
> related to the displacement of the ROI centroid in the field of view (p = 0.290) or to the
> CellReg score (p = 0.344), while centroid displacement was related to CellReg score (p = 0.004).

HD cells valid in the first and last session of the circular protocol. Centroids are the
mask-weighted centres of the ROI image masks, in pixels of 3.3 um (`grid_spacing`, 0.0033 mm).
The figure has four tests (claims file, statistics): shift against displacement over all cells and
over cells with CellReg score above 0.8, shift against score, and displacement against score, so
the Bonferroni factor is 4. "All cells" are taken with every CellReg score (the HD rule without the
score cut); cells with score > 0.8 only are a variant. Readings: pooled over mice or the Fig. 1
example mouse (mouse 4); signed or absolute shift; smoothed or raw HDI.

In [ ]:
REG = {}
for cfg in [MAIN, CONFIGS[1]]:
    rows = []
    for m in MICE15:
        T = H.hd_table(H.group(S, m, "circular"), sigma=cfg[0], hdi_kind=cfg[1], score_min=-np.inf)
        g = T["sessions"]
        jl = len(g) - 1
        cells = np.where(T["valid"][:, 0] & T["valid"][:, jl])[0]
        if len(cells) == 0:
            continue
        sh = H.pd_change(T, 0, jl)
        c0 = g[0]["centroid"][T["R"][cells, 0]]
        c1 = g[jl]["centroid"][T["R"][cells, jl]]
        disp = np.hypot(*(c1 - c0).T) * g[0]["pixel_mm"][0] * 1000
        for i, c in enumerate(cells):
            rows.append(dict(mouse=m, shift_deg=DEG(sh[i]), disp_um=disp[i], score=T["score"][c]))
    reg = pd.DataFrame(rows)
    print("/".join(cfg), ": cells with a CellReg score below 0 (not a valid score), dropped:", int((reg.score < 0).sum()),
          reg[reg.score < 0].score.round(3).tolist())
    REG[cfg] = reg[reg.score >= 0]
for cfg in [MAIN, CONFIGS[1]]:
    reg = REG[cfg]
    print("/".join(cfg), ": cells valid in first and last session per mouse, all scores", reg.groupby("mouse").size().to_dict(),
          "total", len(reg), "; score > 0.8", reg[reg.score > 0.8].groupby("mouse").size().to_dict(), "total", int((reg.score > 0.8).sum()))
    print("   centroid displacement %.1f to %.1f um; score %.2f to %.2f" % (reg.disp_um.min(), reg.disp_um.max(), reg.score.min(), reg.score.max()))
print("expected n = 72")


def edf2_tests(sel, kind):
    y = lambda d: d.shift_deg if kind == "signed" else d.shift_deg.abs()
    hi = sel[sel.score > 0.8]
    out = {}
    for name, (x, yy, d) in {"shift~disp, all": ("disp_um", None, sel), "shift~disp, score>0.8": ("disp_um", None, hi),
                             "shift~score": ("score", None, sel), "disp~score": ("score", "disp_um", sel)}.items():
        if len(d) < 3:
            out[name] = (len(d), np.nan, np.nan)
            continue
        r = stats.linregress(d[x], y(d) if yy is None else d[yy])
        out[name] = (len(d), r.slope, r.pvalue)
    return out


rows = []
for cfg in [MAIN, CONFIGS[1]]:
    reg = REG[cfg]
    for cellset, base in [("all scores", reg), ("score > 0.8 only", reg[reg.score > 0.8])]:
        for label, sel in [("pooled", base), ("mouse 4", base[base.mouse == 4])]:
            for kind in ["signed", "absolute"]:
                t = edf2_tests(sel, kind)
                p4 = {k: min(1, 4 * v[2]) for k, v in t.items()}
                ok = (p4["shift~disp, all"] >= 0.05) and (p4["shift~score"] >= 0.05) and (p4["disp~score"] < 0.05)
                rows.append(dict(reading="/".join(cfg), cells=cellset, sel=label, shift=kind, n=len(sel),
                                 **{f"p {k}": v[2] for k, v in t.items()}, **{f"slope {k}": v[1] for k, v in t.items()},
                                 p_disp_score_x4=p4["disp~score"], matches=ok))
e2 = pd.DataFrame(rows)
print(e2[["reading", "cells", "sel", "shift", "n", "p shift~disp, all", "p shift~disp, score>0.8", "p shift~score", "p disp~score",
          "p_disp_score_x4", "matches"]].round(4).to_string(index=False))
print("\nslopes (degrees per um, degrees per unit score, um per unit score):")
print(e2[["reading", "cells", "sel", "shift", "slope shift~disp, all", "slope shift~score", "slope disp~score"]].round(3).to_string(index=False))
print("expected p 0.290 (shift~disp, all), 0.588 corrected (score > 0.8), 0.344 (shift~score), 0.004 corrected (disp~score)")

**Result: reproduced.** Over all cells (every valid CellReg score), the paper's "all cells", 49 HD cells are valid in the first and last session under the main reading and 106 with raw HDI (the paper has 72). Two raw-HDI cells with a CellReg score below 0 (-0.096 and -0.01), which is not a valid score, are dropped; the main reading has none. Pooled over mice, the two absence tests are not significant (shift against displacement p = 0.074 signed and 0.62 absolute; shift against score p = 0.98 and 0.38), and displacement against score is significant (p = 0.0003, 0.0011 after Bonferroni over 4 tests); with raw HDI the same holds (absence p 0.084 to 0.24, displacement against score p < 0.0001). For mouse 4 alone (10 or 19 cells) displacement against score is not significant (p = 0.96 and 0.94). Restricted to cells with score > 0.8 (28 or 56 cells), displacement against score is not significant after Bonferroni over 4 (0.67 and 0.055), so the claim would not reproduce: the outcome rests on reading "all cells" as every CellReg score, an assumption.

## fig1h-drift-rate

> The absolute change in preferred direction grew slowly with the days between recordings, at a
> drift rate of 7.1 plus or minus 4.4 degrees per week across mice.

Every pair of sessions; the value per pair is (a) the absolute session-level circular mean change
or (b) the median over cells of the absolute change. Drift rate readings of the claims file:
(a) slope over all pairs, (b) slope of the means per day difference, (c) slope through the origin.

In [ ]:
def pair_values(T):
    ns = T["valid"].shape[1]
    dd = H.day_diff(T["sessions"])
    out = []
    for a in range(ns):
        for b in range(a + 1, ns):
            d = H.pd_change(T, a, b)
            if len(d):
                out.append(dict(a=a, b=b, days=dd[a, b], pop=abs(DEG(H.circ_mean(d))), cell=np.median(np.abs(DEG(d)))))
    return pd.DataFrame(out)


PV1 = {cfg: {m: pair_values(CIRC[cfg][m]) for m in MICE15} for cfg in CONFIGS}

In [ ]:
DRIFT_PD = {}
for cfg in CONFIGS:
    for val in ["pop", "cell"]:
        rates = pd.DataFrame({m: H.drift_rates(PV1[cfg][m].days, PV1[cfg][m][val]) for m in MICE15}).T
        DRIFT_PD[(cfg, val)] = rates
        print("%-14s value %-4s per-mouse rates (a) %s (b) %s (c) %s | means a %.2f b %.2f c %.2f"
              % ("/".join(cfg), val, rates.a.round(1).tolist(), rates.b.round(1).tolist(), rates.c.round(1).tolist(),
                 rates.a.mean(), rates.b.mean(), rates.c.mean()))
print("expected mean 7.1 +- 4.4 deg/week (s.d. over mice 4.4)")
print("pairs per mouse:", [len(PV1[MAIN][m]) for m in MICE15], "day differences up to", [int(PV1[MAIN][m].days.max()) for m in MICE15])

**Result: reproduced.** The mean drift is 6.71 degrees per week (reading a, population value; per mouse 6.2, -1.2, 4.3, 14.0, 10.3), within 7.1 +- 4.4. Drift readings a and b fall within it under both value readings; reading c (line through the origin) gives 13.6 to 15.1.

## edf4g-pairwise-offset-stable

> The change in pairwise angular offset across all registered HD cell pairs was strongly biased
> towards 0, with per-animal medians of 12, 9, 12, 24 and 12 degrees against shuffle medians of
> 51 to 54 degrees (p < 0.001 per animal, 25,901 pairs).

In [ ]:
EXP_O = [12, 9, 12, 24, 12]; TOL_O = [27.6, 24.1, 27.1, 40.9, 31.1]
OFFS1, OFFCH1 = {}, {}
for cfg in [MAIN, CONFIGS[1]]:
    rng = np.random.default_rng(3)
    rows = []
    for m in MICE15:
        T = CIRC[cfg][m]
        offs = H.all_offsets(T)
        ns = offs.shape[0]
        pairs = [(a, b) for a in range(ns) for b in range(a + 1, ns)]
        OFFS1[(cfg, m)] = offs
        for uns in [True, False]:
            per, hist = H.offset_changes(offs, pairs, rng, N_SHUF, unsigned=uns)
            OFFCH1[(cfg, m, uns)] = (pairs, per)
            allc = np.concatenate(per)
            shm = H.hist_median(hist)
            n_cell_pairs = int((np.isfinite(offs).sum(0) > 0).sum())
            rows.append(dict(offsets="unsigned" if uns else "signed", mouse=m, median=np.median(allc), sd=np.std(allc),
                             shuffle_median=np.median(shm), shuffle_min=shm.min(), below_all=bool(np.median(allc) < shm.min()),
                             n_values=len(allc), n_cell_pairs=n_cell_pairs))
    df = pd.DataFrame(rows)
    df["within"] = (df["median"] - np.array(EXP_O)[df.mouse - 1]).abs() <= np.array(TOL_O)[df.mouse - 1]
    print("/".join(cfg))
    print(df.round(2).to_string(index=False))
    for o in ["unsigned", "signed"]:
        d = df[df.offsets == o]
        print("%s: within tolerance %d of 5, below all shuffled medians %d of 5; cell pairs %d; cell pair x session pair values %d (expected 25,901 pairs)"
              % (o, d.within.sum(), d.below_all.sum(), d.n_cell_pairs.sum(), d.n_values.sum()))

**Result: reproduced.** With unsigned offsets the per-animal medians are 12, 9, 12, 21 and 12 degrees, all within the paper's s.d., and each is below all 1,000 shuffled medians (shuffle medians 39 to 51 degrees, against the paper's 51 to 54). There are 25,441 cell pair by session pair values against the paper's 25,901. Signed offsets give the same outcome, with shuffle medians of 72 to 90 degrees.

## fig1i-offset-drift

> The change in pairwise angular offset drifted more slowly (4.1 plus or minus 4.8 degrees per
> week) than single cell tuning, though not significantly so.

Per session pair, the median over cell pairs of the absolute change in offset; drift readings as
for fig1h; unsigned offsets (main reading, see edf4g). The comparison with the fig1h rates (value reading (a), the same drift reading) by a
two-sided Wilcoxon signed-rank and a paired t-test across the five mice.

In [ ]:
for cfg in [MAIN, CONFIGS[1]]:
    rates = {}
    for m in MICE15:
        pairs, per = OFFCH1[(cfg, m, True)]
        dd = H.day_diff(CIRC[cfg][m]["sessions"])
        x = [dd[a, b] for (a, b), c in zip(pairs, per) if len(c)]
        y = [np.median(c) for c in per if len(c)]
        rates[m] = H.drift_rates(x, y)
    rates = pd.DataFrame(rates).T
    for r in ["a", "b", "c"]:
        pd_r = DRIFT_PD[(cfg, "pop")][r].values
        w = stats.wilcoxon(rates[r].values, pd_r)
        t = stats.ttest_rel(rates[r].values, pd_r)
        print("%-14s reading %s: offset rates %s mean %.2f s.d. %.2f | PD rates mean %.2f | Wilcoxon W %.1f p %.3f | paired t %.2f p %.3f | mean difference %.2f deg/week"
              % ("/".join(cfg), r, rates[r].round(2).tolist(), rates[r].mean(), rates[r].std(ddof=0), pd_r.mean(), w.statistic, w.pvalue,
                 t.statistic, t.pvalue, (rates[r].values - pd_r).mean()))
print("expected 4.1 +- 4.8 deg/week")

**Result: reproduced.** The offset drift is 5.32 degrees per week (reading a; s.d. 4.62), within 4.1 +- 4.8; reading b gives 5.41 and reading c 12.17. Against the preferred-direction drift (6.71), the Wilcoxon (p = 0.188) and the paired t-test (p = 0.132) are not significant. With raw HDI the drift is 5.46 and both tests are not significant.

## edf4ij-pv-decay

> The population vector coherence decayed slowly over the course of the recording.

Population vectors of the HD cells valid in both sessions, raw tuning curves averaged in 12
head-direction bins. Readings: (a) mean over bins of the correlation across cells, (b)
correlation of the flattened cell by bin matrices.

In [ ]:
for cfg in [MAIN, CONFIGS[1]]:
    for reading, key in [("a", "bin"), ("b", "flat")]:
        n_ok, det = 0, []
        for m in MICE15:
            T = CIRC[cfg][m]
            dd = H.day_diff(T["sessions"])
            ns = len(dd)
            x, y = [], []
            for a in range(ns):
                for b in range(a + 1, ns):
                    v = H.pv_corr(T, a, b, reading=key)
                    if np.isfinite(v):
                        x.append(dd[a, b]); y.append(v)
            x, y = np.array(x), np.array(y)
            rho, p = stats.spearmanr(x, y)
            last = y[x == x.max()].mean()
            ok = rho < 0 and p < 0.05
            n_ok += ok
            det.append("m%d rho %.2f p %.2g, mean at %d days %.2f, at 1 day %.2f" % (m, rho, p, x.max(), last, y[x == 1].mean() if (x == 1).any() else np.nan))
        print("/".join(cfg), "reading", reading, ": negative and p < 0.05 in", n_ok, "of 5")
        for d in det:
            print("   ", d)

**Result: reproduced.** Under both readings, the population vector correlation decreases with day difference in 4 of 5 mice (Spearman p < 0.05; mouse 5 has rho -0.15, p = 0.11, reading a). Its mean at the longest difference stays above 0 in every mouse (0.09 to 0.86). With raw HDI it decreases in 5 of 5 mice under both readings, and the mean at the longest difference of mouse 5 is -0.00 (reading a) and -0.19 (reading b), so that part holds in 4 of 5 there. The outcome therefore rests on the main HD reading (HDI on the smoothed curve).

## edf1h-ring

> An Isomap projection of the HD cell population activity forms a ring, reflecting the circular
> representation of head direction.

One example session per mouse (the session with the most ROIs whose split-half correlation
exceeds 0.8, read in full above for the clock check). Cells with split-half correlation > 0.8
and HDI > 0.1 in the session (main reading); dF/F smoothed with a Gaussian of sigma 133 ms,
normalized (readings: z-score, division by the maximum), averaged in 0.25 s bins, bins whose mean
normalized activity is below 0.1 dropped, Isomap with 50 neighbours to two dimensions (Methods,
Dimensionality reduction). The test (circular-circular correlation of the angle around the
embedding's centroid with head direction) is the claims file's.

In [ ]:
from sklearn.manifold import Isomap


def corrcc(a, b):
    """Circular-circular correlation and its p-value, as pycircstat.corrcc."""
    sa, sb = np.sin(a - H.circ_mean(a)), np.sin(b - H.circ_mean(b))
    r = (sa * sb).sum() / np.sqrt((sa ** 2).sum() * (sb ** 2).sum())
    n = len(a)
    l20, l02, l22 = (sa ** 2).mean(), (sb ** 2).mean(), (sa ** 2 * sb ** 2).mean()
    ts = np.sqrt(n * l20 * l02 / l22) * r
    return r, 2 * (1 - stats.norm.cdf(abs(ts)))


ring = []
fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
for col, m in enumerate(MICE15):
    s = EXAMPLE[m]
    dff, ts, hd, hts = RAW[m]
    cells = np.where((s["split_12deg"] > 0.8) & (s["hdi_s_12deg"] > 0.1))[0]
    for row, norm in enumerate(["zscore", "max"]):
        x = H.gaussian_filter1d(dff[:, cells], 0.133 * s["fs"], axis=0)
        x = (x - x.mean(0)) / x.std(0) if norm == "zscore" else x / x.max(0)
        edges = np.arange(ts[0], ts[-1], 0.25)
        idx = np.searchsorted(edges, ts, side="right") - 1
        ok = (idx >= 0) & (idx < len(edges) - 1)
        cnt = np.bincount(idx[ok], minlength=len(edges) - 1)
        xb = np.zeros((len(edges) - 1, len(cells)))
        np.add.at(xb, idx[ok], x[ok])
        good = cnt > 0
        xb = xb[good] / cnt[good, None]
        centers = (edges[:-1] + 0.125)[good]
        keep = xb.mean(1) >= 0.1
        xb, centers = xb[keep], centers[keep]
        hb = H.circ_interp(centers, hts, np.mod(hd, 2 * np.pi))
        if len(xb) <= 50:
            ring.append(dict(mouse=m, normalization=norm, session=f"{s['protocol']} {s['k']}", cells=len(cells),
                             bins=len(xb), r=np.nan, p=np.nan, radius_cv=np.nan))
            axes[row, col].set_title(f"mouse {m}, {norm}: {len(xb)} bins", fontsize=9)
            continue
        emb = Isomap(n_neighbors=50, n_components=2).fit_transform(xb)
        ang = np.arctan2(*(emb - emb.mean(0)).T[::-1])
        r, p = corrcc(ang, hb)
        rad = np.hypot(*(emb - emb.mean(0)).T)
        ring.append(dict(mouse=m, normalization=norm, session=f"{s['protocol']} {s['k']}", cells=len(cells),
                         bins=len(xb), r=r, p=p, radius_cv=rad.std() / rad.mean()))
        ax = axes[row, col]
        ax.scatter(emb[:, 0], emb[:, 1], c=hb, cmap="hsv", s=2)
        ax.set_title(f"mouse {m}, {norm}, r={r:.2f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
ring = pd.DataFrame(ring)
print(ring.round(4).to_string(index=False))
for norm in ["zscore", "max"]:
    d = ring[ring.normalization == norm]
    print(norm, ": significant (p < 0.05) in", int((d.p < 0.05).sum()), "of 5; |r| from %.2f to %.2f; sessions with too few bins for 50 neighbours: %d"
          % (d.r.abs().min(), d.r.abs().max(), d.r.isna().sum()))
print("all embedded points are plotted; colour is head direction (0 to 2 pi)")

**Result: reproduced.** With z-scored activity, the angle around the Isomap embedding is correlated with head direction in all five example sessions (circular correlation p < 0.05, |r| from 0.19 to 0.81). With division by the maximum, four of the five sessions keep fewer than 51 bins above the 0.1 threshold, too few for 50 neighbours. The p values treat the 0.25 s bins as independent, which they are not, so they overstate the evidence; mouse 1's correlation (r = -0.19) is weak evidence of a ring.

## Multi-environment protocol

40 sessions per mouse (37 for mouse 2, 35 for mouse 5), two weeks in each of the circular, square,
8-arm maze and open field environments, one CellReg table per mouse. HD cells as for Fig. 1
(registered in >= 6 sessions, score > 0.8), main reading.

In [ ]:
MULTI = tables("multi")
MULTI_RAW = tables("multi", CONFIGS[1])
for m in MICE15:
    g = MULTI[m]["sessions"]
    envs = [s["env"] for s in g]
    print("mouse %d: %d sessions, %s; HD cells %d (raw HDI %d), median per session %d"
          % (m, len(g), ", ".join(f"{e} {envs.count(e)}" for e in dict.fromkeys(envs)), MULTI[m]["valid"].any(1).sum(),
             MULTI_RAW[m]["valid"].any(1).sum(), np.median(MULTI[m]["valid"].sum(0))))
EPOCH2 = {"Circular": 1, "Square": 1, "8-arm maze": 2, "Open field": 2}  # mouse 2: Miniscope changed after the square

## fig2e-offset-across-environments

> Across the four environments, HD cells preserved their pairwise angular offsets, with
> per-animal median changes of 24, 12, 18, 29 and 12 degrees against shuffle medians of 54 to 57
> degrees, and a drift rate of 1.9 plus or minus 1.02 degrees per week.

In [ ]:
EXP_2E = [24, 12, 18, 29, 12]; TOL_2E = [38.9, 38.0, 31.5, 42.7, 30.5]
res2e = {}
for cfg, TT in [(MAIN, MULTI), (CONFIGS[1], MULTI_RAW)]:
    rng = np.random.default_rng(4)
    for uns in [True, False]:
        rows = []
        for m in MICE15:
            T = TT[m]
            offs = H.all_offsets(T)
            g = T["sessions"]
            ns = len(g)
            dd = H.day_diff(g)
            for variant in (["all", "within epoch"] if m == 2 else ["all"]):
                pairs = [(a, b) for a in range(ns) for b in range(a + 1, ns)
                         if variant == "all" or EPOCH2[g[a]["env"]] == EPOCH2[g[b]["env"]]]
                per, hist = H.offset_changes(offs, pairs, rng, N_SHUF if cfg == MAIN else 200, unsigned=uns)
                allc = np.concatenate(per)
                shm = H.hist_median(hist)
                x = [dd[a, b] for (a, b), c in zip(pairs, per) if len(c)]
                y = [np.median(c) for c in per if len(c)]
                dr = H.drift_rates(x, y)
                rows.append(dict(mouse=m, pairs=variant, median=np.median(allc), sd=np.std(allc), shuffle_median=np.median(shm),
                                 below_all=bool(np.median(allc) < shm.min()), n=len(allc), rate_a=dr["a"], rate_b=dr["b"], rate_c=dr["c"]))
        df = pd.DataFrame(rows)
        res2e[(cfg, uns)] = df
        print("/".join(cfg), "unsigned" if uns else "signed")
        print(df.round(2).to_string(index=False))
        for label, sel in [("(a) mouse 2 excluded", df[(df.mouse != 2)]),
                           ("(b) mouse 2 included", df[df.pairs == "all"]),
                           ("(c) mouse 2 within epoch", df[(df.mouse != 2) | (df.pairs == "within epoch")])]:
            within = sum(abs(r["median"] - EXP_2E[r.mouse - 1]) <= TOL_2E[r.mouse - 1] for _, r in sel.iterrows())
            print("   %-26s medians within %d of %d, below all shuffles %d; mean drift a %.2f b %.2f c %.2f"
                  % (label, within, len(sel), sel.below_all.sum(), sel.rate_a.mean(), sel.rate_b.mean(), sel.rate_c.mean()))
print("expected medians", EXP_2E, "drift 1.9 +- 1.02 deg/week")

**Result: reproduced.** With unsigned offsets the medians are 24, 9, 15, 27 and 12 degrees, all within the paper's s.d., and all are below every shuffled median (51 to 57). The mean drift is 1.79 (reading b, mouse 2 included) or 2.00 degrees per week (mouse 2 excluded), within 1.9 +- 1.02; reading a gives 2.28 and 2.41, also within, and reading c gives 6.9, outside.

## edf5c-stable-within-environment

> The tuning of HD cells was stable within each environment, with changes in preferred direction
> relative to the last day concentrated around 0 degrees in all four, and no difference between
> environments (repeated-measures ANOVA, F = 0.44, P = 0.73).

In [ ]:
from statsmodels.stats.anova import AnovaRM
ENVS = ["Circular", "Square", "8-arm maze", "Open field"]
EXP_5C = [44.5, 15.5, 35.4, 31.5]
for cfg, TT in [(MAIN, MULTI), (CONFIGS[1], MULTI_RAW)]:
    per_env, per_me = {e: [] for e in ENVS}, []
    for m in MICE15:
        T = TT[m]
        g = T["sessions"]
        for e in ENVS:
            js = [j for j, s in enumerate(g) if s["env"] == e]
            ref = js[-2] if (m == 2 and e == "Square") else js[-1]
            ch = []
            for j in js:
                if j == ref:
                    continue
                d = H.pd_change(T, ref, j)
                if len(d):
                    ch.append(H.circ_mean(d))
            per_env[e] += ch
            a = np.abs(DEG(np.array(ch)))
            per_me.append(dict(mouse=m, env=e, n=len(ch), mean_abs=a.mean() if len(a) else np.nan, median_abs=np.median(a) if len(a) else np.nan))
    print("/".join(cfg))
    for e, x in zip(ENVS, EXP_5C):
        V, p = H.vtest(np.array(per_env[e]))
        print("   %-11s n %2d  V %.2f (expected %.1f +- %.2f)  one-sided p %.2g" % (e, len(per_env[e]), V, x, 0.1 * x, p))
    pm = pd.DataFrame(per_me)
    print(pm.pivot(index="mouse", columns="env", values="n"))
    for val in ["mean_abs", "median_abs"]:
        if pm[val].isna().any():
            print("   ANOVA", val, ": missing cells", pm[pm[val].isna()][["mouse", "env"]].values.tolist())
            continue
        a = AnovaRM(pm, val, "mouse", within=["env"]).fit().anova_table
        print("   repeated-measures ANOVA on %s: F %.2f, df %d/%d, p %.3f" % (val, a["F Value"].iloc[0], a["Num DF"].iloc[0], a["Den DF"].iloc[0], a["Pr > F"].iloc[0]))
        print("   per-environment means of %s:" % val, pm.groupby("env")[val].mean().round(1).to_dict())

**Result: reproduced.** V = 44.18, 13.81, 33.76 and 31.51 over 45, 39, 44 and 43 sessions (the paper's n), all p < 0.001; 3 of 4 are within 10% (square, 13.81 against 15.5 +- 1.55, is not). The repeated-measures ANOVA is not significant (F = 1.01, p = 0.42 on means; F = 0.84, p = 0.50 on medians).

## Remapping between environments (fig2i, fig2h)

For each pair of recordings of two environments, the shift of every HD cell valid in both
(cross-correlation), its circular mean (remapping) and mean vector length (coherency); the
average remapping of a mouse and environment pair is the circular mean over recording pairs
(Methods, Orientation remapping). Head direction is in room coordinates for the circular, square
and 8-arm maze (`reference_frame` of the files). Shuffles permute cell identity in the second
recording, 1,000 times.

In [ ]:
ENV_PAIRS = [("Circular", "Square"), ("Square", "8-arm maze"), ("8-arm maze", "Circular")]
OF_PAIRS = [("Circular", "Open field"), ("Square", "Open field"), ("8-arm maze", "Open field")]


def remapping(TT, rng):
    out = []
    for m in MICE15:
        T = TT[m]
        g = T["sessions"]
        for e1, e2 in ENV_PAIRS + OF_PAIRS:
            j1 = [j for j, s in enumerate(g) if s["env"] == e1]
            j2 = [j for j, s in enumerate(g) if s["env"] == e2]
            for a in j1:
                for b in j2:
                    cells, M = H.shift_matrix(T, a, T, b)
                    if len(cells) < 2:
                        continue
                    own = np.diag(M)
                    _, sh_r = H.shuffled_means(M, rng, N_SHUF)
                    out.append(dict(mouse=m, pair=f"{e1} to {e2}", a=a, b=b, n=len(cells), remap=H.circ_mean(own),
                                    mvl=H.resultant(own), shuffle_p95=np.percentile(sh_r, 95), shuffle_median=np.median(sh_r)))
    out = pd.DataFrame(out)
    return out.groupby(["mouse", "pair"]).agg(remap=("remap", lambda x: DEG(H.circ_mean(x.values))), rec_pairs=("remap", "size"),
                                              median_cells=("n", "median"), mvl=("mvl", "median"), p95=("shuffle_p95", "median"),
                                              shuf_med=("shuffle_median", "median")).reset_index()


AVG = {MAIN: remapping(MULTI, np.random.default_rng(5)), CONFIGS[1]: remapping(MULTI_RAW, np.random.default_rng(5))}
for cfg in AVG:
    print("/".join(cfg))
    print(AVG[cfg].round(2).to_string(index=False))

## fig2i-random-remapping

> The remapping of the HD system between the three environments of one room seemed random across
> mice (Rayleigh test, P = 0.36 circular to square, 0.61 square to radial maze, 0.032 radial maze
> to circular).

In [ ]:
for cfg in AVG:
    avg = AVG[cfg]
    print("/".join(cfg))
    for e1, e2 in ENV_PAIRS:
        d = avg[avg.pair == f"{e1} to {e2}"]
        z, p = H.rayleigh(np.radians(d.remap.values))
        print("   %-24s mice %d, average remapping %s deg, Rayleigh z %.2f p %.3f" % (f"{e1} to {e2}", len(d), d.remap.round(0).tolist(), z, p))
print("expected z 1.08, 0.53, 3.19; p 0.36, 0.61, 0.032")

**Result: partially reproduced.** Rayleigh z = 1.28 (p = 0.29), 0.70 (p = 0.52) and 2.58 (p = 0.069). The first two are not significant, as in the paper. The third is not significant either, where the paper reports p = 0.032, and no z is within 10% of the paper's. Raw HDI gives the same pattern: z = 1.31 (p = 0.28), 0.45 (p = 0.66) and 2.66 (p = 0.063).

## fig2h-coherent-remapping

> While the internal structure of the HD system was largely kept across environments, its
> absolute orientation was specific to each environment, the HD cells of a mouse shifting together
> (coherent remapping) compared with shuffled cell identities.

In [ ]:
for cfg in AVG:
    avg = AVG[cfg]
    print("/".join(cfg))
    for label, pairs in [("room environments", ENV_PAIRS), ("open field pairs (variant)", OF_PAIRS)]:
        d = avg[avg.pair.isin([f"{a} to {b}" for a, b in pairs])]
        coh = d.mvl > d.p95
        print("   %s: coherent in %d of %d mouse and environment pairs (%.0f%%); not coherent: %s"
              % (label, coh.sum(), len(d), 100 * coh.mean(), [f"m{r.mouse} {r.pair}" for _, r in d[~coh].iterrows()]))
        big = d.assign(big=d.remap.abs() > 30).groupby("pair").big.sum()
        print("      mice with |average remapping| > 30 deg per pair:", big.to_dict())

**Result: not reproduced.** Under the main reading, remapping is coherent (median mean vector length above the median shuffle 95th percentile) in 10 of 15 mouse and environment pairs, 67% against the criterion's 80%. The absolute average remapping exceeds 30 degrees in at least 3 mice for every environment pair. Mouse 2 contributes few recording pairs across its Miniscope change (2 for 8-arm maze to circular). With raw HDI, which keeps about twice as many cells per recording pair, 12 of 15 pairs are coherent (80%, at the threshold), with remapping above 30 degrees in at least 3 mice for every pair, which meets the criterion. The outcome rests on the HDI reading.

## fig2g-environment-classifier

> Classifiers of environment identity trained on pairwise correlations of HD cell tuning curves
> were more accurate than shuffles (proportion correct 0.7 against 0.3; one-sided Wilcoxon,
> P = 0.031, W = 15).

Per mouse, one feature per pair of HD cells: the correlation of their tuning curves in the
recording, missing (NaN, handled by XGBoost) where either cell is not valid in it, an assumption
(the claims file leaves open how absent cells enter). Training on a random half of the
recordings of each environment, testing on the rest; shuffles permute the features within each
recording. Readings: mean over 100 random splits, and a single split; main and raw HDI.

Controls, main reading. Which cell pairs are present follows the session date, and each
environment is visited in a two-week block, so a classifier could use detection, not the
correlations. (i) Presence only: each feature set to 1 where present, missing elsewhere.
(ii) Split in time: per environment, train on the recordings of the first half of its block and
test on the second half, and the reverse; the mean of the two. (iii) Values shuffled: within each
recording the correlations are permuted among the pairs present in it, so the presence pattern is
kept and only the values are destroyed; the data are compared with this shuffle.

In [ ]:
import xgboost as xgb


def env_features(T):
    g = T["sessions"]
    cells = np.where(T["valid"].any(1))[0]
    i, j = np.triu_indices(len(cells), 1)
    X = np.full((len(g), len(i)), np.nan, dtype=np.float32)
    for k, s in enumerate(g):
        v = T["valid"][cells, k]
        tc = np.stack([tc_of_safe(T, c, k) for c in cells])
        C = np.corrcoef(tc)
        ok = v[i] & v[j]
        X[k, ok] = C[i[ok], j[ok]]
    y = np.array([ENVS.index(s["env"]) for s in g])
    return X, y


def tc_of_safe(T, c, k):
    r = T["R"][c, k]
    return T["sessions"][k][f"tc_{T['sigma']}"][r].astype(float) if r >= 0 else np.zeros(H.NBINS)


def fit_accuracy(X, y, tr, te, rng, shuffle):
    Xs = X.copy()
    if shuffle == "values":
        for k in range(len(Xs)):
            ok = np.where(np.isfinite(Xs[k]))[0]
            Xs[k, ok] = Xs[k, ok[rng.permutation(len(ok))]]
    elif shuffle:
        for k in range(len(Xs)):
            Xs[k] = Xs[k][rng.permutation(Xs.shape[1])]
    clf = xgb.XGBClassifier(n_jobs=4, random_state=0)
    clf.fit(Xs[tr], y[tr])
    return (clf.predict(Xs[te]) == y[te]).mean()


def split_accuracy(X, y, rng, shuffle):
    tr = np.concatenate([rng.permutation(np.where(y == e)[0])[: (y == e).sum() // 2] for e in range(4)])
    te = np.setdiff1d(np.arange(len(y)), tr)
    return fit_accuracy(X, y, tr, te, rng, shuffle)


def time_split_accuracy(X, y, rng, shuffle):
    first = np.concatenate([np.where(y == e)[0][: (y == e).sum() // 2] for e in range(4)])
    second = np.setdiff1d(np.arange(len(y)), first)
    return np.mean([fit_accuracy(X, y, first, second, rng, shuffle), fit_accuracy(X, y, second, first, rng, shuffle)])


def wilcoxon_line(acc, a, b, label):
    w = stats.wilcoxon(acc[a], acc[b], alternative="greater")
    print("   %-34s %.2f (s.d. %.2f) vs shuffle %.2f (s.d. %.2f); one-sided Wilcoxon W %.1f p %.3f"
          % (label, acc[a].mean(), acc[a].std(ddof=0), acc[b].mean(), acc[b].std(ddof=0), w.statistic, w.pvalue))


ACC = {}
for cfg, TT in [(MAIN, MULTI), (CONFIGS[1], MULTI_RAW)]:
    rows = []
    t_start = time.time()
    for m in MICE15:
        X, y = env_features(TT[m])
        rng = np.random.default_rng(100 + m)
        data = [split_accuracy(X, y, rng, False) for _ in range(100)]
        shuf = [split_accuracy(X, y, rng, True) for _ in range(100)]
        rng1 = np.random.default_rng(7)
        one = split_accuracy(X, y, rng1, False)
        one_s = split_accuracy(X, y, rng1, True)
        row = dict(mouse=m, recordings=len(y), features=X.shape[1], finite=float(np.isfinite(X).mean()),
                   data_mean=np.mean(data), shuffle_mean=np.mean(shuf), single_data=one, single_shuffle=one_s)
        if cfg == MAIN:
            Xp = np.where(np.isfinite(X), 1, np.nan).astype(np.float32)
            rngc = np.random.default_rng(200 + m)
            row["presence_mean"] = np.mean([split_accuracy(Xp, y, rngc, False) for _ in range(100)])
            row["presence_shuffle"] = np.mean([split_accuracy(Xp, y, rngc, True) for _ in range(100)])
            rngv = np.random.default_rng(400 + m)
            row["values_shuffle"] = np.mean([split_accuracy(X, y, rngv, "values") for _ in range(100)])
            rngt = np.random.default_rng(300 + m)
            row["time_split"] = time_split_accuracy(X, y, rngt, False)
            row["time_split_shuffle"] = np.mean([time_split_accuracy(X, y, rngt, True) for _ in range(10)])
        rows.append(row)
    acc = pd.DataFrame(rows)
    ACC[cfg] = acc
    print("/".join(cfg), "(%.0f s)" % (time.time() - t_start))
    print(acc.round(3).to_string(index=False))
    wilcoxon_line(acc, "data_mean", "shuffle_mean", "mean over 100 splits")
    wilcoxon_line(acc, "single_data", "single_shuffle", "single split")
    if cfg == MAIN:
        wilcoxon_line(acc, "presence_mean", "presence_shuffle", "control (i), presence only")
        wilcoxon_line(acc, "time_split", "time_split_shuffle", "control (ii), split in time")
        wilcoxon_line(acc, "data_mean", "values_shuffle", "control (iii), correlations vs values shuffled")
print("expected 0.7 +- 0.1 vs 0.3 +- 0.1, W 15, p 0.031; chance with 4 environments 0.25")

**Result: reproduced.** The mean proportion correct over 100 splits is 0.64 (s.d. 0.04) against 0.28 for shuffles, one-sided Wilcoxon W = 15, p = 0.031. A single split gives 0.70 against 0.21, with the same test result. With raw HDI the values are 0.59 against 0.26 and 0.63 against 0.28, also W = 15, p = 0.031. The controls do not support reading this as an environment code in the correlations: a classifier given only which cell pairs are present in each recording reaches 0.70 (shuffle 0.28), as high as with the correlations, and training and testing on different halves of each environment's block gives 0.48 (shuffle 0.27; still W = 15, p = 0.031). Shuffling only the values, among the pairs present in each recording, gives 0.59 against 0.64 for the data (W = 15, p = 0.031; lower in 5 of 5 mice), so the correlations add a little to what the presence pattern carries.

## edf9-interference

> After four further weeks of visits to the four environments in random order, the system
> showed reduced retention of the original preferred direction, though changes relative to week 1
> stayed centred on 0 degrees (V = 11.3, 10.1, 10.8), and pairwise offsets drifted at a mean of
> 1.6 degrees per week.

The protocol "4 environments are visited weekly in a random order, for 4 weeks" holds 16 sessions
per mouse (15 for mice 2 and 5). Its first week follows the last day of the two-week protocol and
visits each environment once, so it is the probe week of Fig. 3, and weeks 2 to 4 are those of
Extended Data Fig. 9. Weeks are counted from the first session (7-day blocks). The HD cell rule
for this figure is not stated: the Methods set two sessions for Figs. 3 and 4 and six for Figs. 1
and 2. Both are run; two is the main reading (an assumption: the same protocol as Fig. 3, with at
most four visits per environment). CellReg score > 0.8. Main and raw HDI. The first cell prints
the dates on which the probe week is identified.

In [ ]:

# The random-order protocol against the end of the two-week (multi) protocol.
for m in MICE15:
    gm, gr = H.group(S, m, "multi"), H.group(S, m, "random")
    dm, dr = H.session_dates(gm), H.session_dates(gr)
    wk1 = [s["env"] for s, d in zip(gr, dr) if (d - dr[0]).days < 7]
    print("mouse %d: last two-week session %s (%s), first random session %s, gap %d days; first week of the random protocol: %d sessions, %s"
          % (m, dm[-1], gm[-1]["env"], dr[0], (dr[0] - dm[-1]).days, len(wk1), ", ".join(wk1)))


In [ ]:
for cfg, min_s in [(MAIN, 2), (MAIN, 6), (CONFIGS[1], 2), (CONFIGS[1], 6)]:
    RND = tables("random", cfg, min_sessions=min_s)
    weekly = {2: [], 3: [], 4: []}
    rates, wk4 = [], []
    rng = np.random.default_rng(8)
    for m in MICE15:
        T = RND[m]
        g = T["sessions"]
        d0 = H.session_dates(g)
        week = np.array([(d - d0[0]).days // 7 + 1 for d in d0])
        for e in ENVS:
            js = [j for j, s in enumerate(g) if s["env"] == e]
            w1 = [j for j in js if week[j] == 1]
            if not w1:
                continue
            for j in js:
                if week[j] == 1:
                    continue
                d = H.pd_change(T, w1[0], j)
                if len(d):
                    weekly[week[j]].append(H.circ_mean(d))
                    if week[j] == 4:
                        wk4.append(abs(DEG(H.circ_mean(d))))
        offs = H.all_offsets(T)
        ns = len(g)
        dd = H.day_diff(g)
        pairs = [(a, b) for a in range(ns) for b in range(a + 1, ns)]
        per, _ = H.offset_changes(offs, pairs, unsigned=True)
        x = [dd[a, b] for (a, b), c in zip(pairs, per) if len(c)]
        y = [np.median(c) for c in per if len(c)]
        rates.append(H.drift_rates(x, y))
    rates = pd.DataFrame(rates, index=MICE15)
    print("%s, sessions registered in >= %d:" % ("/".join(cfg), min_s))
    for w, x in zip([2, 3, 4], [11.3, 10.1, 10.8]):
        V, p = H.vtest(np.array(weekly[w]))
        print("   week %d: n %d, V %.2f (expected %.1f +- %.2f), one-sided p %.2g" % (w, len(weekly[w]), V, x, 0.1 * x, p))
    print("   offset drift per mouse (deg/week) a %s b %s c %s; means a %.2f b %.2f c %.2f (expected 1.6 +- 0.16)"
          % (rates.a.round(2).tolist(), rates.b.round(2).tolist(), rates.c.round(2).tolist(), rates.a.mean(), rates.b.mean(), rates.c.mean()))
    print("   mean absolute change at week 4 relative to week 1: %.1f deg over %d values (the 4-week probe value of fig3de is not available, see the summary)"
          % (np.mean(wk4), len(wk4)))

**Result: partially reproduced.** The probe week is the first week of the random-order protocol: for mice 1 to 4 it starts 3 days after the last two-week session; for mouse 5 the gap is 81 days. V = 10.38, 8.91 and 10.56 over 19, 19 and 18 values (the paper has 20, 20, 18), all with p < 0.01, so the changes relative to week 1 stay centred on 0; week 3 is outside 10.1 +- 1.01. The pairwise offset drift is 0.68 degrees per week (reading a; b gives -0.11, c gives 11.70), against 1.6 +- 0.16. Raw HDI gives the same (V 10.24, 8.68, 10.71; drift -0.52). Reduced retention could not be compared, as the 4-week probe value of fig3de is blocked; the mean absolute change at week 4 is 42.2 degrees.

## Single-visit protocol (Fig. 4, Extended Data Fig. 10)

Mice 6 to 10, seven sessions: both environments on day 1, one of them weekly, both on week 5.
HD cells registered in >= 2 sessions with CellReg score > 0.8 (Methods). The weekly environment is
the one with five sessions.

In [ ]:
MICE610 = [6, 7, 8, 9, 10]
SINGLE = {MAIN: tables("single", min_sessions=2), CONFIGS[1]: tables("single", CONFIGS[1], min_sessions=2)}
FLS = {}
for cfg in SINGLE:
    FL = []
    rng = np.random.default_rng(9)
    for m in MICE610:
        T = SINGLE[cfg][m]
        g = T["sessions"]
        envs = [s["env"] for s in g]
        for e in sorted(set(envs)):
            js = [j for j, s in enumerate(g) if s["env"] == e]
            group = "weekly" if len(js) > 2 else "single"
            cells, M = H.shift_matrix(T, js[0], T, js[-1])
            own = np.diag(M)
            shm, _ = H.shuffled_means(M, rng, N_SHUF)
            FL.append(dict(mouse=m, env=e, group=group, sessions=len(js), cells=len(cells), change=DEG(H.circ_mean(own)),
                           abs_change=abs(DEG(H.circ_mean(own))), shuffle_median=np.median(np.abs(DEG(shm))),
                           shuffle_mean=np.mean(np.abs(DEG(shm)))))
    FLS[cfg] = pd.DataFrame(FL)
    print("/".join(cfg))
    print(FLS[cfg].round(2).to_string(index=False))

## fig4d-single-visit-memory

> After a single visit, the orientation of the HD cell population was retained four weeks later
> as well as with weekly visits; the change in preferred direction from first to last session was
> smaller than shuffles in both groups (one-sided Wilcoxon, P = 0.031) and did not differ between
> groups (P = 0.438; weekly 12.4, single 6.6 degrees).

In [ ]:
for cfg, FL in FLS.items():
    print("/".join(cfg))
    for red in ["shuffle_median", "shuffle_mean"]:
        for grp in ["weekly", "single"]:
            d = FL[FL.group == grp].sort_values("mouse")
            w = stats.wilcoxon(d.abs_change, d[red], alternative="less")
            print("   %-14s %-6s: change %s vs shuffle %s; one-sided Wilcoxon W %.1f p %.3f"
                  % (red, grp, d.abs_change.round(1).tolist(), d[red].round(1).tolist(), w.statistic, w.pvalue))
    wk = FL[FL.group == "weekly"].sort_values("mouse").abs_change.values
    sg = FL[FL.group == "single"].sort_values("mouse").abs_change.values
    w = stats.wilcoxon(wk, sg)
    print("   weekly vs single: two-sided Wilcoxon W %.1f p %.3f; means %.1f (s.d. %.1f) and %.1f (s.d. %.1f); expected 12.4 +- 10.0 and 6.6 +- 4.1"
          % (w.statistic, w.pvalue, wk.mean(), wk.std(), sg.mean(), sg.std()))
    print("   variant without mouse 8: means %.1f and %.1f" % (np.delete(wk, 2).mean(), np.delete(sg, 2).mean()))

**Result: partially reproduced.** In both groups the change is smaller than in the shuffles (one-sided Wilcoxon W = 0, p = 0.031), and the groups do not differ (two-sided p = 0.188). The weekly mean, 6.2 degrees, is within 12.4 +- 10.0. The single-visit mean, 21.9 degrees, is outside 6.6 +- 4.1, raised by mouse 8 (48 degrees, from 2 cells) and mouse 10 (30 degrees). With raw HDI the tests are the same (W = 0, p = 0.031 in both groups; group test W = 4, p = 0.438, the paper's values), the weekly mean is 8.3 (within) and the single mean 12.4 (outside 6.6 +- 4.1). Under both readings the two means would fit if the groups were swapped.

## edf10c-environment-identity

> The change in preferred direction from first to last visit did not differ between the two
> environments (circular mean 1.6 against 5.1 degrees; two-sided Wilcoxon, p = 1.0, n = 10).

The sign of a change in preferred direction is not fixed by the Methods ("the angular shift
corresponding to the maximum correlation value"). As computed, a change from session a to a later
session b is the preferred direction in b minus that in a. The paper's per-mouse means of Fig. 1g
(15.4, 2.1, -5.5, -7.8, -6.9, change of each session relative to the last) have the same sign as
the values computed here in 5 of 5 mice, and those are the earlier session minus the later one.
The same convention here is the first visit minus the later one, the sign-reversed values. Both
signs are printed; the earlier minus the later is the main reading, an assumption.

In [ ]:
for cfg, FL in FLS.items():
    for sign, label in [(-1, "first visit minus later (main)"), (1, "later minus first visit (as computed)")]:
        pv = FL.pivot(index="mouse", columns="env", values="change") * sign
        w = stats.wilcoxon(pv["Circular"], pv["Square"])
        means = {}
        for e in ["Circular", "Square"]:
            x = np.radians(pv[e].values)
            means[e] = (DEG(H.circ_mean(x)), DEG(np.sqrt(-2 * np.log(H.resultant(x)))))
        ok1 = abs(means["Circular"][0] - 1.6) <= 14.1 and abs(means["Square"][0] - 5.1) <= 8.0
        ok2 = abs(means["Circular"][0] - 5.1) <= 8.0 and abs(means["Square"][0] - 1.6) <= 14.1
        print("%s, %s: per mouse circular %s, square %s" % ("/".join(cfg), label, pv["Circular"].round(1).tolist(), pv["Square"].round(1).tolist()))
        print("   circular mean %.1f (s.d. %.1f), square %.1f (s.d. %.1f); within under assignment circular 1.6 / square 5.1: %s, circular 5.1 / square 1.6: %s; two-sided Wilcoxon W %.1f p %.3f"
              % (*means["Circular"], *means["Square"], ok1, ok2, w.statistic, w.pvalue))
print("expected 1.6 +- 14.1 and 5.1 +- 8.0 (assignment open), W 7, p 1.0")
print("weekly environment per mouse:", FLS[MAIN][FLS[MAIN].group == "weekly"].set_index("mouse").env.to_dict())

**Result: reproduced.** Under the main sign reading (first visit minus later), the circular means are -0.2 degrees (circular) and 12.8 (square), both within tolerance under either assignment, and the two-sided Wilcoxon is not significant (W = 3, p = 0.312), as in the paper. With the sign as first computed (later minus first) square is -12.8, outside under both assignments. With raw HDI, the means are -4.4 and 1.2 (main sign) or 4.4 and -1.2, within tolerance under at least one assignment, and p = 0.625. The outcome rests on the sign convention, an assumption.

## edf10b-per-animal-change

> Per animal of the single-visit protocol, the circular mean change in preferred direction
> relative to the first visit was 4.7, -0.4, -15.5, 5.4 and -6.2 degrees.

Sign readings as for edf10c-environment-identity.

In [ ]:
EXP_10B = [4.7, -0.4, -15.5, 5.4, -6.2]; TOL_10B = [3.4, 5.8, 34.7, 10.8, 10.9]
for cfg in SINGLE:
    rows = []
    for m, x, t in zip(MICE610, EXP_10B, TOL_10B):
        T = SINGLE[cfg][m]
        g = T["sessions"]
        ch = []
        for e in sorted(set(s["env"] for s in g)):
            js = [j for j, s in enumerate(g) if s["env"] == e]
            for j in js[1:]:
                d = H.pd_change(T, js[0], j)
                if len(d):
                    ch.append(H.circ_mean(d))
        ch = np.array(ch)
        cm = DEG(H.circ_mean(ch))
        rows.append(dict(mouse=m, n=len(ch), first_minus_later=-cm, later_minus_first=cm,
                         circ_sd=DEG(np.sqrt(-2 * np.log(H.resultant(ch)))), expected=x, tol=t,
                         within_main=abs(-cm - x) <= t, within_reversed=abs(cm - x) <= t))
    d10 = pd.DataFrame(rows)
    print("/".join(cfg))
    print(d10.round(2).to_string(index=False))
    print("   within tolerance: %d of 5 (first minus later, main), %d of 5 (later minus first); signs as the paper's: %d and %d of 5"
          % (d10.within_main.sum(), d10.within_reversed.sum(), (np.sign(d10.first_minus_later) == np.sign(EXP_10B)).sum(),
             (np.sign(d10.later_minus_first) == np.sign(EXP_10B)).sum()))

**Result: reproduced.** Under the main sign reading (first visit minus later), the circular means are 3.25, -3.07, -12.43, 4.68 and -7.71 degrees: all 5 within tolerance and all with the paper's sign. With raw HDI also 5 of 5 (5.27, -4.86, -12.03, 7.31, -8.29). With the sign as first computed, 3 of 5 (main) and 2 of 5 (raw) are within tolerance. The outcome rests on the sign convention, an assumption.

## edf7-ephys-offsets

> Electrophysiological recordings confirmed that pairwise angular offsets of HD cells are
> preserved across environments, with per-animal median changes of 18, 12, 27 and 15 degrees
> against shuffle medians of 54 degrees (p < 0.001).

Units on the shanks whose electrode group is located in PoSub ("PSB"). Each file holds two
explorations of each environment (separated by sleep); the paper describes one, so the main
reading is the first exploration of each, and all six are a variant. HD cell: mean rate > 1 Hz
and HDI > 0.3 (Methods), the rate and the HDI (on the smoothed curve, as the imaging main
reading; raw HDI shown) averaged over the three first explorations; the offsets are
computed with either cell set. Tuning curves as for imaging
(3 degree bins, 12 degree kernel), the claims file's reading. Offsets unsigned (main reading, see
edf4g).

In [ ]:
EXP_7 = [18, 12, 27, 15]; TOL_7 = [41.7, 35.5, 43.2, 39.5]
rows, counts = [], []
rng = np.random.default_rng(10)
for d in EP:
    psb = [i for i, gname in enumerate(d["group_names"]) if d["group_location"][gname] == "PSB"]
    wake = [e for e in d["epochs"] if "wake" in e[2]]
    first = []
    for e in wake:
        if e[2][1] not in [f[2][1] for f in first]:
            first.append(e)
    tcs, sms, hdis_s, hdis_r, rates = [], [], [], [], []
    for (t0, t1, tags) in first:
        tc, sm, occ, rt = H.spike_tuning([d["spikes"][i] for i in psb], t0, t1, d["hd_t"], d["hd"])
        tcs.append(tc); sms.append(sm); rates.append(rt)
        hdis_s.append(H.hdi(sm, occ)); hdis_r.append(H.hdi(tc, occ))
    rate = np.mean(rates, 0)
    hd_s = (rate > 1) & (np.mean(hdis_s, 0) > 0.3)
    hd_r = (rate > 1) & (np.mean(hdis_r, 0) > 0.3)
    counts.append(dict(mouse=d["mouse"], units=len(d["spikes"]), psb_units=len(psb), hd_smoothed_hdi=int(hd_s.sum()),
                       hd_raw_hdi=int(hd_r.sum()), first_explorations=[f[2][1] for f in first]))
    for hdi_label, cells in [("smoothed HDI", np.where(hd_s)[0]), ("raw HDI", np.where(hd_r)[0])]:
        for label, eps in [("first explorations", first), ("all explorations", wake)]:
            offs = []
            for (t0, t1, tags) in eps:
                _, sm, _, _ = H.spike_tuning([d["spikes"][psb[i]] for i in cells], t0, t1, d["hd_t"], d["hd"])
                o = np.full((len(cells), len(cells)), np.nan, dtype=np.float32)
                a, b, val = H.pair_offsets(sm)
                o[a, b] = val
                offs.append(o)
            offs = np.array(offs)
            pairs = [(a, b) for a in range(len(eps)) for b in range(a + 1, len(eps)) if eps[a][2][1] != eps[b][2][1]]
            for uns in [True, False]:
                per, hist = H.offset_changes(offs, pairs, rng, N_SHUF, unsigned=uns)
                allc = np.concatenate(per)
                shm = H.hist_median(hist)
                rows.append(dict(mouse=d["mouse"], cells=hdi_label, explorations=label, offsets="unsigned" if uns else "signed", hd_cells=len(cells),
                                 median=np.median(allc), sd=np.std(allc), shuffle_median=np.median(shm), below_all=bool(np.median(allc) < shm.min())))
print(pd.DataFrame(counts).to_string(index=False))
print("HD cells, four animals: %d (smoothed HDI), %d (raw HDI); expected 125 +- 12.5; non-HD PoSub units %d (smoothed)"
      % (sum(c["hd_smoothed_hdi"] for c in counts), sum(c["hd_raw_hdi"] for c in counts),
         sum(c["psb_units"] - c["hd_smoothed_hdi"] for c in counts)))
e7 = pd.DataFrame(rows)
e7["expected"] = [EXP_7[int(m.split()[-1]) - 1] for m in e7.mouse]
e7["within"] = [abs(r["median"] - EXP_7[int(r.mouse.split()[-1]) - 1]) <= TOL_7[int(r.mouse.split()[-1]) - 1] for _, r in e7.iterrows()]
print(e7.round(2).to_string(index=False))

**Result: reproduced.** With unsigned offsets over the first explorations, the per-animal medians are 6, 6, 9 and 6 degrees, within the paper's s.d., and each is below all 1,000 shuffled medians (51 to 57). The four files hold 254 PoSub units, the paper's n. Of these, 99 are HD cells under smoothed HDI and 118 under raw HDI, against the paper's 125 +- 12.5. With the raw-HDI cells the medians are 6, 9, 9 and 6 degrees, also below all shuffles; signed offsets and all six explorations give the same outcome under both cell sets. The medians are one third to one half of the paper's (18, 12, 27, 15).

In [ ]:
# Clock check for the electrophysiology: median HDI of PoSub units in the first exploration with
# spikes shifted against head direction.
for d in EP:
    psb = [i for i, gname in enumerate(d["group_names"]) if d["group_location"][gname] == "PSB"]
    t0, t1, _ = [e for e in d["epochs"] if "wake" in e[2]][0]
    out = []
    for sh in [-2, -0.5, -0.1, 0, 0.1, 0.5, 2]:
        _, sm, occ, _ = H.spike_tuning([d["spikes"][i] + sh for i in psb], t0, t1, d["hd_t"], d["hd"])
        out.append("%+.1f s: %.3f" % (sh, np.nanmedian(H.hdi(sm, occ))))
    print(d["mouse"], "; ".join(out))

## Summary

| Claim | Outcome | Rests on an assumption |
|---|---|---|
| roi-counts | partially reproduced (402 sessions, not 388; ROI range and mean match) | no |
| edf1e-hd-proportion | not reproduced | HD rule readings, none matches |
| edf1h-ring | reproduced (z-score normalization) | normalization |
| fig1f-example-stable | reproduced (mouse 2) | HDI on the smoothed curve |
| fig1g-all-mice-stable | reproduced | HDI on the smoothed curve |
| edf4ab-vtest-per-animal | reproduced | HDI on the smoothed curve |
| edf4c-previous-session | reproduced | HDI on the smoothed curve |
| edf4f-change-below-shuffle | partially reproduced | no (no reading meets both parts) |
| edf4ef-not-diffusive | reproduced (reading a) | HDI on the smoothed curve |
| edf2ab-jaccard | not reproduced | HDI on the smoothed curve (raw HDI reproduces it) |
| edf2-no-registration-confound | reproduced (all CellReg scores, pooled) | "all cells" as every CellReg score (score > 0.8 only: not reproduced); both HDI readings agree |
| fig1h-drift-rate | reproduced | drift rate definition |
| edf4g-pairwise-offset-stable | reproduced | no (signed and unsigned offsets agree) |
| fig1i-offset-drift | reproduced | unsigned offsets, drift definition |
| edf4ij-pv-decay | reproduced | HDI on the smoothed curve (raw HDI: decay in 5 of 5, but mouse 5's mean at the longest interval is 0 or below) |
| fig1k-decoding-across-days | blocked, too_large (decoding.ipynb gives the estimate, 80 min of fits) | |
| fig2e-offset-across-environments | reproduced | unsigned offsets, drift definition |
| edf5c-stable-within-environment | reproduced | no |
| fig2i-random-remapping | partially reproduced | no (raw HDI the same) |
| fig2h-coherent-remapping | not reproduced | HDI on the smoothed curve (raw HDI: 80%, reproduced) |
| fig2g-environment-classifier | reproduced (cell-pair presence alone classifies as well; correlations add a little) | missing pairs as missing features |
| edf7-ephys-offsets | reproduced | no (both offsets and both cell sets agree) |
| edf7g-waveforms | blocked, data_missing | |
| edf8-same-vs-mixed | partially reproduced (decoding.ipynb) | |
| fig3de-probe-memory, fig3e-mvl-delay, fig3fg-probe-drift | blocked, metadata_insufficient | |
| edf9-interference | partially reproduced (V-tests significant; drift and one V outside) | unsigned offsets, minimum of two sessions, weeks from the protocol's start |
| fig4d-single-visit-memory | partially reproduced | no (raw HDI the same) |
| fig4e-decoder-single-visit | reproduced (decoding.ipynb) | |
| edf10c-environment-identity | reproduced | sign of the change (first visit minus later) |
| edf10b-per-animal-change | reproduced | sign of the change (first visit minus later) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's code is not public, so several details
were taken as assumptions: the 12 degree kernel; HDI on the smoothed tuning curve, which gives
the paper's numbers of compared sessions and offset values, while raw HDI gives about twice as
many HD cells; unsigned pairwise offsets, whose shuffle medians (39 to 57 degrees here) are close
to the paper's 51 to 57, while signed offsets give 72 to 90; drift rates as least-squares slopes; "detected" read as
registered by CellReg; the sign of a change in preferred direction as the earlier session minus
the later one, the convention under which the paper's signs of Fig. 1g and Extended Data Fig. 10b
are matched. The edf9 HD rule (two sessions) and the decoder's cell rule (HD in both sessions)
are readings of the Methods. Every HD cell claim except edf1h-ring is also run with raw HDI; it
changes the outcome of edf2ab-jaccard, fig2h-coherent-remapping and edf4c-previous-session, and
it fails part of fig1f-example-stable (no mouse with 18 sessions matches) and of edf4ij-pv-decay
(mouse 5's mean at the longest interval is 0 or below). edf2-no-registration-confound rests on
reading "all cells" as every CellReg score; with score > 0.8 only it is not reproduced. Fig. 1k was not run for
time. The environment classifier of Fig. 2g does as well from which cell pairs are present in a
recording as from their correlations; shuffling the correlations among the pairs present lowers it
from 0.64 to 0.59, so the correlations add a little to the presence pattern.

What was found about the dataset:

- The `tuning_curves` table of every imaging file holds one curve repeated for every ROI. It is
  the curve of ROI 0, smoothed with a kernel of about 4 bins (401 of 402 sessions).
- Each protocol has its own CellReg table, and nothing links ROIs across protocols. The probe
  week of Fig. 3 is the first week of the protocol "4 environments are visited weekly in a random
  order", so its cells cannot be matched to the last visits of the two-week protocol.
- The electrophysiology files hold spike times without waveforms; Extended Data Fig. 7f-h cannot
  be tested. They hold 254 PoSub units (the paper's n) and also cortex and CA1 units, and each
  environment was explored twice, where the paper describes one exploration.
- The dataset holds 402 imaging sessions (367 for mice 1 to 5), where the paper reports 388. It
  holds random-order sessions for mouse 5, whose field of view the paper says was lost before
  the probe trials. Sessions last 902 to 1799 s, where the Methods say 20 min.
- The CellReg tables, dates and session numbers are consistent, and dF/F and head direction share
  a clock: head-direction tuning peaks within 0.2 s of zero shift in every checked session (five imaging sessions and the four electrophysiology files).
- The dandiset's related publication is the bioRxiv preprint, not the Nature article.
- The CellReg scores include values below 0 (102 of 5,190 rows, down to -1.24), which a CellReg
  confidence score cannot take; the files do not describe the score. Such cells are dropped where
  every score is used (edf2-no-registration-confound).

In [ ]:
print("notebook run time %.1f min" % ((time.time() - T0) / 60))